# 🎙️ Student 1 — Audio Analyst: 911 Call Transcription & Analysis

**Assignment:** AI for Engineers — Multimodal Crime / Incident Report Analyzer  
**Role:** Student 1 — Audio Analyst  
**Pipeline:** Transcribe 911 audio → Extract keywords/entities → Sentiment & urgency scoring → Export CSV

### ✅ Required Output Columns
`Call_ID | Transcript | Extracted_Event | Location | Sentiment | Urgency_Score`

---

## How this notebook works
1. **Cells 1–3** — Fork from Kaggle Wav2Vec2 notebook; installs & transcription already done there. This notebook uses **Whisper** as an equivalent local approach with synthetic 911 audio.
2. **Cells 4–6** — Your own analysis cells (keyword extraction, sentiment, urgency).
3. **Cell 7** — Save the final CSV.

> **Kaggle users:** Run the Wav2Vec2 fork first, then paste Cells 4–7 at the bottom of that notebook. Replace `transcripts` with whatever variable the Wav2Vec2 notebook produces.

In [ ]:
# ============================================================
# CELL 1 — Install all dependencies
# ============================================================
# Run this once. On Kaggle with GPU this takes ~2-3 minutes.

!pip install openai-whisper spacy transformers torch torchaudio soundfile -q
!python -m spacy download en_core_web_sm -q

print("✅ All packages installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 83.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ All packages installed.


In [ ]:
# ============================================================
# CELL 2 — Detect real 911 audio files from audio_files/
# ============================================================
# Scans audio_files/ for .wav files and confirms they exist.

import os
import glob

audio_dir = "."
wav_files = sorted(glob.glob(os.path.join(audio_dir, "*.wav")))

assert len(wav_files) > 0, f"No .wav files found in {audio_dir}/"

# Build call list: [(call_id, filepath), ...]
calls = [(os.path.splitext(os.path.basename(f))[0], f) for f in wav_files]

print(f"Found {len(calls)} audio file(s) in ./{audio_dir}/:\n")
for call_id, path in calls:
    size_kb = os.path.getsize(path) / 1024
    print(f"  {call_id}: {path}  ({size_kb:.1f} KB)")
print("\n✅ Audio files confirmed. Ready for transcription.")

Found 5 audio file(s) in ././:

  C001: ./C001.wav  (1033.6 KB)
  C002: ./C002.wav  (1033.6 KB)
  C003: ./C003.wav  (1033.6 KB)
  C004: ./C004.wav  (1033.6 KB)
  C005: ./C005.wav  (1033.6 KB)

✅ Audio files confirmed. Ready for transcription.


In [ ]:
# ============================================================
# CELL 3 — Transcribe real audio using OpenAI Whisper
# ============================================================
# Uses soundfile to load WAV (no ffmpeg required).

import whisper
import numpy as np
import soundfile as sf

model = whisper.load_model("base")
print("Whisper model loaded. Transcribing real 911 audio...\n")

transcripts = {}
for call_id, filepath in calls:
    data, sr = sf.read(filepath, dtype='float32')
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != 16000:
        import scipy.signal
        data = scipy.signal.resample(data, int(len(data) * 16000 / sr))
    audio = whisper.pad_or_trim(data)
    mel = whisper.log_mel_spectrogram(audio, model.dims.n_mels).to(model.device)
    options = whisper.DecodingOptions(fp16=False)
    result = whisper.decode(model, mel, options)
    transcripts[call_id] = result.text.strip()
    print(f"{call_id}: {transcripts[call_id]}")

print(f"\nTranscribed {len(transcripts)} calls.")


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 277MiB/s]


Whisper model loaded. Transcribing real 911 audio...

C001: Okay, we've got a fire down here at the Route 7 right here at Abington.
C002: I don't know but you have 4,000 good to drive. Really take the left at that light.
C003: No, I'm not somebody. You just broke into our house. We're all alone. Please help us. That's somebody's in the house now. I'm not.
C004: Hi, I'm Karen St. and I'm not doing a lot of interviews with my house.
C005: Are you out of the house yet? No. Or I get out of the house. We can't.

Transcribed 5 calls.


In [ ]:
# ============================================================
# CELL A — spaCy: keyword & named-entity extraction
# ============================================================
# Extracts locations (GPE / LOC / FAC) and classifies incident
# type using rule-based keyword matching.

import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")

KEYWORD_MAP = [
    (["fire", "flame", "smoke", "burning", "blaze"],           "Fire / Building fire"),
    (["accident", "crash", "collision", "vehicle", "car"],     "Road accident"),
    (["robbery", "theft", "stolen", "gun", "weapon", "steal"], "Robbery / Theft"),
    (["fight", "assault", "attack", "violence", "attacking"],  "Assault / Fight"),
    (["trapped", "injured", "hurt", "medical", "unconscious"], "Medical Emergency"),
]

def classify_event(text):
    text_lower = text.lower()
    for keywords, label in KEYWORD_MAP:
        if any(w in text_lower for w in keywords):
            return label
    return "General incident"

def extract_location(text):
    doc = nlp(text)
    locs = [ent.text for ent in doc.ents if ent.label_ in ("GPE", "LOC", "FAC")]
    return locs[0] if locs else "Unknown"

# Build the base dataframe
rows = []
for call_id, transcript in transcripts.items():
    rows.append({
        "Call_ID":         call_id,
        "Transcript":      transcript[:300],          # cap at 300 chars for readability
        "Extracted_Event": classify_event(transcript),
        "Location":        extract_location(transcript),
    })

audio_df = pd.DataFrame(rows)
print("📋 Extraction complete:")
print(audio_df[["Call_ID", "Extracted_Event", "Location"]].to_string(index=False))

📋 Extraction complete:
Call_ID      Extracted_Event Location
   C001 Fire / Building fire  Route 7
   C002     General incident  Unknown
   C003     General incident  Unknown
   C004     General incident  Unknown
   C005     General incident  Unknown


In [ ]:
# ============================================================
# CELL B — Sentiment & urgency analysis
# ============================================================
# Sentiment: DistilBERT fine-tuned on SST-2 (NEGATIVE = Distressed)
# Urgency:   keyword hit-count + sentiment confidence score

from transformers import pipeline

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

URGENCY_WORDS = [
    "help", "emergency", "hurry", "dying", "trapped", "please",
    "now", "quickly", "fire", "weapon", "injured", "immediately",
    "robbery", "attacking", "shot"
]

sentiments = []
urgencies  = []

for _, row in audio_df.iterrows():
    text = row["Transcript"][:512]          # DistilBERT token limit
    result = sentiment_pipe(text)[0]

    # Sentiment label
    sentiment = "Distressed" if result["label"] == "NEGATIVE" else "Calm"

    # Urgency score: keyword hits (0.1 each) + model confidence (up to 0.4), capped at 1.0
    hit_count = sum(1 for w in URGENCY_WORDS if w in text.lower())
    urgency = round(min(0.2 + (hit_count * 0.1) + result["score"] * 0.3, 1.0), 2)

    sentiments.append(sentiment)
    urgencies.append(urgency)
    print(f"{row['Call_ID']} → Sentiment: {sentiment:10s} | Urgency: {urgency:.2f} | Hits: {hit_count}")

audio_df["Sentiment"]    = sentiments
audio_df["Urgency_Score"] = urgencies

print("\n✅ Sentiment and urgency scoring complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

C001 → Sentiment: Distressed | Urgency: 0.59 | Hits: 1
C002 → Sentiment: Calm       | Urgency: 0.59 | Hits: 1
C003 → Sentiment: Distressed | Urgency: 0.80 | Hits: 3
C004 → Sentiment: Distressed | Urgency: 0.50 | Hits: 0
C005 → Sentiment: Distressed | Urgency: 0.50 | Hits: 0

✅ Sentiment and urgency scoring complete.


In [ ]:
# ============================================================
# CELL C — Validate & save audio_output.csv
# ============================================================

REQUIRED_COLUMNS = ["Call_ID", "Transcript", "Extracted_Event", "Location", "Sentiment", "Urgency_Score"]

# Ensure correct column order
audio_df = audio_df[REQUIRED_COLUMNS]

# Validate
assert list(audio_df.columns) == REQUIRED_COLUMNS, f"Column mismatch! Got: {list(audio_df.columns)}"
assert len(audio_df) > 0, "DataFrame is empty!"

audio_df.to_csv("audio_output.csv", index=False)

print("✅ Saved audio_output.csv")
print(f"   Rows: {len(audio_df)} | Columns: {list(audio_df.columns)}\n")
print(audio_df.to_string(index=False))

✅ Saved audio_output.csv
   Rows: 5 | Columns: ['Call_ID', 'Transcript', 'Extracted_Event', 'Location', 'Sentiment', 'Urgency_Score']

Call_ID                                                                                                                         Transcript      Extracted_Event Location  Sentiment  Urgency_Score
   C001                                                            Okay, we've got a fire down here at the Route 7 right here at Abington. Fire / Building fire  Route 7 Distressed           0.59
   C002                                                 I don't know but you have 4,000 good to drive. Really take the left at that light.     General incident  Unknown       Calm           0.59
   C003 No, I'm not somebody. You just broke into our house. We're all alone. Please help us. That's somebody's in the house now. I'm not.     General incident  Unknown Distressed           0.80
   C004                                                             Hi, I'm Karen St.

In [ ]:
# ============================================================
# CELL D — Download (Colab only)
# ============================================================
# On Kaggle: use the folder icon on the left sidebar to find
# audio_output.csv and download it manually.
# On Google Colab: uncomment the lines below.

# from google.colab import files
# files.download("audio_output.csv")

print("📥 On Kaggle: click the 📁 folder icon (left sidebar) → find audio_output.csv → download.")
print("📥 On Colab:  uncomment the two lines above.")

📥 On Kaggle: click the 📁 folder icon (left sidebar) → find audio_output.csv → download.
📥 On Colab:  uncomment the two lines above.


---
## 📤 GitHub Upload Instructions

1. **Download `audio_output.csv`** — Kaggle: folder icon → file → Download  
2. **Download this notebook** — Kaggle: `File → Download Notebook`  
3. **Go to your GitHub repo** → open the `/audio` folder  
4. Click **Add file → Upload files**  
5. Drag both `audio_output.csv` and this `.ipynb` file → **Commit changes**

### ✅ Final CSV Column Checklist
| Column | Type | Example |
|---|---|---|
| `Call_ID` | string | `C001` |
| `Transcript` | string (≤300 chars) | `There is a fire...` |
| `Extracted_Event` | string | `Fire / Building fire` |
| `Location` | string | `Downtown Ave` |
| `Sentiment` | `Distressed` / `Calm` | `Distressed` |
| `Urgency_Score` | float 0.0–1.0 | `0.91` |